![Astrofisica Computacional](../new_logo.png)

# Computational Astrophysics – `Astroquery`

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.



### About this notebook

In this worksheet we will present some basic examples of the use of `Astroquery`. 

---

### Example 01

We use `Astroquery` to query [Vizier](http://vizier.cfa.harvard.edu) about a specific keyword, and we will use of [astropy.coordinates](https://docs.astropy.org/en/stable/coordinates/index.html) to restrict the results. 

Vizier’s keywords can indicate features such as wavelength or object type. You can see some of the keywords directly at Vizier's webpage, [https://vizier.cds.unistra.fr/viz-bin/VizieR](https://vizier.cds.unistra.fr/viz-bin/VizieR). 

You can try some of these (IT IS IMPORTANT TO USE DOUBLE QUOTES " "):

- "stars:white_dwarf"
- "Black_Holes"
- "Supernovae"
- "QSOs"
- "Gravitational_wave"
- "Pulsars"

In [ ]:
from astroquery.vizier import Vizier
from astropy import coordinates
from astropy import units as u

# Defining the class of the objects

v = Vizier(keywords=["stars:white_dwarf", "optical"])
v

In [ ]:
# Define a reference point in the sky using coordinates
c = coordinates.SkyCoord(0, 0, unit=('deg', 'deg'), frame='icrs')

# Restrict the query to thos objects of the desired object in a certain region
# around our reference point
result = v.query_region(c, radius=0.5*u.deg)

# Number of datasets in the restricted query
len(result)

In [ ]:
# The complete query includes these tables
# Note the bib-reference of each table
result

In [ ]:
# A dataset (table) in the results is accesed via
result[23]

---

### Example 02

We will look into [SIMBAD](http://simbad.u-strasbg.fr/simbad/) for a particular object.



In [ ]:
from astroquery.simbad import Simbad

result_table = Simbad.query_object("m1")

In [ ]:
result_table.pprint(show_unit=True)

---

### Example 03

We will look again into [SIMBAD](http://simbad.u-strasbg.fr/simbad/) for items in a particular range of timer and restric for a particular object.


In [ ]:
from astroquery.simbad import Simbad

s = Simbad()
# bibcodelist(date1-date2) lists the number of bibliography
# items referring to each object over that date range
s.add_votable_fields('bibcodelist')
s

In [ ]:
# Restrict for a particular object
result2 = s.query_object('m31')
result2

---

### Example 04 

This illustrates finding the spectral type of some particular star.



In [ ]:
from astroquery.simbad import Simbad

customSimbad = Simbad()

# We add the type of the object
customSimbad.add_votable_fields('sp_type')

We restric the query to the object [g Her](http://simbad.u-strasbg.fr/simbad/sim-id?Ident=g+Her)

In [ ]:
# Restrict to the star 
result = customSimbad.query_object('g her')
result

In [ ]:
# ID of the object
result['main_id'][0]

In [ ]:
# Spectral Type of the object
result['sp_type'][0]

### Example 05

We will search into the [open exoplanet catalogue](http://www.openexoplanetcatalogue.com)  to find the mass of a specific planet.
 

In [ ]:
from astroquery import open_exoplanet_catalogue as oec
from astroquery.open_exoplanet_catalogue import findvalue

cata = oec.get_catalogue()

# Specify the exoplanet
kepler68b = cata.find(".//planet[name='Kepler-68 b']")

kepler68b

In [ ]:
# The mass of the planet is
findvalue(kepler68b, 'mass')

---

## Accesing the SDSS database using a SQL query

We will access the SDSS database through astroquery and permform a SQL call,

In [ ]:
from astroquery.sdss import SDSS

#query = "SELECT top 10 z, ra, dec, bestObjID FROM  specObj WHERE  class = 'galaxy' AND z > 0.3  AND zWarning = 0"

q1 = "SELECT top 10 z, ra, dec, bestObjID "
q2 = "FROM  specObj "
q3 = "WHERE  class = 'galaxy' AND z > 0.3  AND zWarning = 0"

query = q1 + q2 + q3


results = SDSS.query_sql(query)
results

---

## SQL query in the SDSS database and crossmatch with other catalogues

We make a SQL query to the SDSS and use `XMatch` to cross match with other catalogues, e.g. with WISE

In [ ]:
from astroquery.sdss import SDSS

sdss = SDSS()

#query = "select top 100 z, ra, dec, bestObjID from specObj where class = 'galaxy' and z > 0.3 and zWarning = 0"

q1 = "SELECT top 100 z, ra, dec, bestObjID "
q2 = "FROM  specObj "
q3 = "WHERE  class = 'galaxy' AND z > 0.3  AND zWarning = 0"

query = q1 + q2 + q3


results = sdss.query_sql(query)
results

In [ ]:
from astroquery.xmatch import XMatch
from astropy import units as u

xmatch = XMatch()

SDSS_WISE_match = xmatch.query(cat1=results,
                               cat2='vizier:ii/328/allwise',
                               max_distance=3 * u.arcsec, colRA1='ra',colDec1='dec')

SDSS_WISE_match